# Batch Evaluation Metrics

Objetivo deste notebook: calcular metricas depois que a inferencia final ja gerou `results_csv`.

Pre-condicao obrigatoria:
- Para cada dataset que voce rodar aqui, precisa existir `evaluation/<dataset>/results_csv/<dataset>_predictions.csv`.
- Se o arquivo nao existir, volte para `batch_eval_table.ipynb`, coloque `RUN_BATCH_INFERENCE=True` e rode a celula daquele dataset.

O que este notebook faz:
- le o CSV agregado de predicoes por dataset
- calcula intent accuracy e macro-F1 de intent usando `semantic_intent` vs `pred_intent`
- reconstroi o gold BIO a partir do dataset bruto original
- calcula metricas de slot em formato CoNLL; o summary principal usa macro por intent, e as colunas `micro_*` preservam o micro global
- exporta relatorios em `evaluation/<dataset>/metrics`, uma celula por dataset


In [ ]:
# Celula 1 - bootstrap/imports.
# Execute uma vez no inicio. Nao carrega modelos; apenas importa pandas e helpers de metricas.

import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / 'src').exists() and (candidate / 'datasets').exists():
            return candidate
    raise RuntimeError('Nao encontrei a raiz do projeto a partir do diretorio atual.')


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
from IPython.display import display as ipy_display

from src.data_tools.batch_eval_utils import load_semantic_alias_registry
from src.data_tools.batch_metrics_utils import (
    compute_batch_metrics_reports,
    export_metrics_reports,
    load_prediction_results_csvs,
)

print('REPO_ROOT =', REPO_ROOT)

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_colwidth', 200)


In [ ]:
# Celula 2 - configuracao de metricas.
# Rode antes das celulas por dataset. Ajuste filtros somente se quiser avaliar um subset especifico.

REGISTRY_PATH = REPO_ROOT / 'datasets' / 'metadata' / 'semantic_alias_registry.json'
PREVIEW_ROWS = 5

# Filtros opcionais. Deixe [] para usar tudo no CSV de predicoes carregado.
# FILTER_DATASET_NAMES filtra coluna dataset_name; normalmente fica [].
FILTER_DATASET_NAMES = []
# FILTER_SOURCE_FILES permite medir apenas arquivos/intents especificos pelo caminho source_file.
FILTER_SOURCE_FILES = []
# FILTER_SEMANTIC_INTENTS permite medir apenas alguns intents semanticos.
FILTER_SEMANTIC_INTENTS = []

# True salva os resumos em evaluation/<dataset>/metrics.
EXPORT_REPORTS = True
# True tambem salva exemplos linha-a-linha; pode gerar CSV maior, entao fica False por padrao.
EXPORT_EXAMPLE_DEBUG = False


In [ ]:
# Celula 3 - funcao de metricas por dataset.
# Sempre execute antes das celulas SNIPS/MultiWOZ/ATIS/Field Operations.
# Esta funcao espera o arquivo evaluation/<dataset>/results_csv/<dataset>_predictions.csv.
# Ela nao roda inferencia; apenas calcula metricas e exporta relatorios.
# Intent e slot F1 principais sao macro por intent; colunas micro_* ficam para auditoria.

def run_metrics_for_dataset(
    dataset_name: str,
    *,
    csv_paths=None,
    filter_dataset_names=None,
    filter_source_files=None,
    filter_semantic_intents=None,
    export_reports: bool = EXPORT_REPORTS,
    export_example_debug: bool = EXPORT_EXAMPLE_DEBUG,
):
    registry = load_semantic_alias_registry(REGISTRY_PATH)
    csv_paths = csv_paths or [
        REPO_ROOT / 'evaluation' / dataset_name / 'results_csv' / f'{dataset_name}_predictions.csv'
    ]
    output_dir = REPO_ROOT / 'evaluation' / dataset_name / 'metrics'

    results_view = load_prediction_results_csvs(
        csv_paths,
        dataset_names=filter_dataset_names if filter_dataset_names is not None else FILTER_DATASET_NAMES,
        source_files=filter_source_files if filter_source_files is not None else FILTER_SOURCE_FILES,
        semantic_intents=filter_semantic_intents if filter_semantic_intents is not None else FILTER_SEMANTIC_INTENTS,
    )

    if results_view.empty:
        raise ValueError(f'Nenhuma linha restante para {dataset_name} apos aplicar filtros do results_csv.')

    print('=' * 80)
    print(f'Metrics dataset: {dataset_name}')
    print('CSV(s) carregados:', len(csv_paths))
    print('Linhas apos filtros:', len(results_view))
    print('Datasets:', sorted(results_view['dataset_name'].drop_duplicates().tolist()))
    print('Source files:', results_view['source_file'].nunique())

    ipy_display(results_view.head(PREVIEW_ROWS))

    reports = compute_batch_metrics_reports(results_view, registry=registry)
    evaluation_examples = reports['evaluation_examples']
    overall_summary = reports['overall_summary']
    intent_summary = reports['intent_summary']
    intent_by_label = reports['intent_by_label']
    intent_confusion = reports['intent_confusion']
    slot_conll_summary = reports['slot_conll_summary']
    slot_conll_by_label = reports['slot_conll_by_label']
    slot_conll_by_intent = reports['slot_conll_by_intent']

    print('Evaluation examples:', evaluation_examples.shape)
    print('Intent errors:', int((~evaluation_examples['intent_correct']).sum()))
    print('Slot chunk mismatches:', int((~evaluation_examples['slot_chunk_exact']).sum()))

    ipy_display(
        evaluation_examples[
            [
                'dataset_name',
                'dataset_intent',
                'semantic_intent',
                'pred_intent',
                'intent_correct',
                'slot_chunk_exact',
                'sentence',
            ]
        ].head(PREVIEW_ROWS)
    )

    print('Overall summary')
    ipy_display(overall_summary)

    print('Intent summary')
    ipy_display(intent_summary)

    print('Intent by label')
    ipy_display(intent_by_label)

    print('Slot CoNLL summary')
    ipy_display(slot_conll_summary)

    print('Slot CoNLL by label')
    ipy_display(slot_conll_by_label)

    print('Slot CoNLL by intent')
    ipy_display(slot_conll_by_intent)

    if not intent_confusion.empty:
        print('Intent confusion matrix')
        ipy_display(intent_confusion)

    intent_errors = evaluation_examples.loc[
        ~evaluation_examples['intent_correct'],
        ['dataset_name', 'dataset_intent', 'semantic_intent', 'pred_intent', 'sentence'],
    ].head(PREVIEW_ROWS)

    slot_errors = evaluation_examples.loc[
        ~evaluation_examples['slot_chunk_exact'],
        [
            'dataset_name',
            'dataset_intent',
            'semantic_intent',
            'pred_intent',
            'sentence',
            'gold_slots_semantic_json_rebuilt',
            'pred_slots_semantic_json_normalized',
        ],
    ].head(PREVIEW_ROWS)

    print('Intent error samples')
    ipy_display(intent_errors)

    print('Slot mismatch samples')
    ipy_display(slot_errors)

    written = {}
    if export_reports:
        written = export_metrics_reports(
            reports,
            output_dir=output_dir,
            include_examples=export_example_debug,
        )
        print('Arquivos exportados:')
        for name, path in written.items():
            print(f'- {name}: {path}')
    else:
        print('EXPORT_REPORTS=False: relatorios nao foram salvos em CSV.')

    return {
        'dataset_name': dataset_name,
        'csv_paths': csv_paths,
        'results_view': results_view,
        'reports': reports,
        'written': written,
    }


## SNIPS

Proposito: calcular metricas apenas do SNIPS.

Pre-condicao: `evaluation/snips/results_csv/snips_predictions.csv` deve existir e corresponder a rodada que voce quer avaliar.


In [ ]:
# Celula SNIPS metrics - rode somente depois de gerar snips_predictions.csv.
snips_metrics = run_metrics_for_dataset('snips')


## MultiWOZ

Proposito: calcular metricas apenas do MultiWOZ.

Pre-condicao: `evaluation/multiwoz/results_csv/multiwoz_predictions.csv` deve existir e corresponder a rodada que voce quer avaliar.


In [ ]:
# Celula MultiWOZ metrics - rode somente depois de gerar multiwoz_predictions.csv.
multiwoz_metrics = run_metrics_for_dataset('multiwoz')


## ATIS

Proposito: calcular metricas apenas do ATIS Min50.

Pre-condicao: `evaluation/atis/results_csv/atis_predictions.csv` deve existir e corresponder a rodada que voce quer avaliar.


In [ ]:
# Celula ATIS metrics - rode somente depois de gerar atis_predictions.csv.
atis_metrics = run_metrics_for_dataset('atis')


## ATIS Stratified Macro@N

Proposito: avaliar a robustez do ATIS sob amostragem balanceada por intent, no estilo Macro@N do `run_pipeline.ipynb` antigo.

Pre-condicao: rode primeiro a celula `atis_metrics = run_metrics_for_dataset('atis')`.

Esta etapa nao roda inferencia e nao altera `evaluation/atis/metrics`. Os CSVs estratificados sao salvos em `evaluation/atis/metrics_stratified/n50_k30_seed42/`.


In [ ]:
# Celula opcional - ATIS Stratified Macro@N.
# Rode depois da celula ATIS metrics. Usa apenas as predicoes ja carregadas em atis_metrics.
# A parte de intent replica o espirito do run_pipeline antigo; a parte de slot e uma extensao de auditoria.

from math import sqrt
from statistics import mean, pstdev


from src.data_tools.batch_metrics_utils import (
    compute_intent_accuracy_reports,
    compute_slot_conll_reports,
)


ATIS_STRAT_N = 50
ATIS_STRAT_K = 30
ATIS_STRAT_SEED = 42
ATIS_STRAT_OUTPUT_DIR = REPO_ROOT / 'evaluation' / 'atis' / 'metrics_stratified' / 'n50_k30_seed42'


def _atis_strat_t_critical_95(df: int) -> float:
    table = {
        1: 12.706,
        2: 4.303,
        3: 3.182,
        4: 2.776,
        5: 2.571,
        6: 2.447,
        7: 2.365,
        8: 2.306,
        9: 2.262,
        10: 2.228,
        12: 2.179,
        15: 2.131,
        20: 2.086,
        25: 2.060,
        30: 2.042,
    }
    if df <= 10:
        return table[df]
    if df <= 30:
        return table[max(key for key in table if key <= df)]
    return 1.96


def _atis_strat_mean_ci95(values):
    values = [float(value) for value in values]
    if not values:
        return 0.0, 0.0, 0.0
    avg = mean(values)
    if len(values) == 1:
        return avg, avg, avg
    sample_std = pstdev(values) * sqrt(len(values) / (len(values) - 1))
    half_width = _atis_strat_t_critical_95(len(values) - 1) * sample_std / sqrt(len(values))
    return avg, avg - half_width, avg + half_width


def _atis_stratified_sample_by_intent(evaluation_examples, n: int, round_index: int, seed: int):
    sampled_parts = []
    for group_index, (_semantic_intent, group) in enumerate(evaluation_examples.groupby('semantic_intent', sort=True)):
        sample_n = min(n, len(group))
        sampled_parts.append(
            group.sample(
                n=sample_n,
                replace=False,
                random_state=seed + round_index * 1009 + group_index,
            )
        )
    if not sampled_parts:
        return evaluation_examples.iloc[0:0].copy()
    return pd.concat(sampled_parts, ignore_index=True)


def run_atis_stratified_macro_analysis(
    *,
    n: int = ATIS_STRAT_N,
    k: int = ATIS_STRAT_K,
    seed: int = ATIS_STRAT_SEED,
    output_dir: Path = ATIS_STRAT_OUTPUT_DIR,
):
    if 'atis_metrics' not in globals():
        raise RuntimeError("Rode primeiro: atis_metrics = run_metrics_for_dataset('atis').")

    evaluation_examples = atis_metrics['reports']['evaluation_examples'].copy()
    if evaluation_examples.empty:
        raise ValueError('atis_metrics nao contem evaluation_examples para estratificar.')

    output_dir.mkdir(parents=True, exist_ok=True)

    support = (
        evaluation_examples.groupby('semantic_intent', sort=True)
        .size()
        .rename('support')
        .reset_index()
    )
    coverage = support.copy()
    coverage['effective_per_round'] = coverage['support'].clip(upper=n)
    coverage['total_sampled_across_rounds'] = coverage['effective_per_round'] * k
    coverage['N'] = n
    coverage['K'] = k
    coverage['seed'] = seed

    round_rows = []
    intent_by_label_frames = []
    slot_by_intent_frames = []

    for round_index in range(k):
        round_number = round_index + 1
        sampled = _atis_stratified_sample_by_intent(evaluation_examples, n=n, round_index=round_index, seed=seed)

        intent_summary, intent_by_label, _intent_confusion = compute_intent_accuracy_reports(sampled)
        slot_summary, _slot_by_label, slot_by_intent = compute_slot_conll_reports(sampled)

        intent_row = intent_summary.iloc[0]
        slot_row = slot_summary.iloc[0]
        round_rows.append(
            {
                'round': round_number,
                'rows': len(sampled),
                'N': n,
                'K': k,
                'seed': seed,
                'intent_count': int(intent_row['intent_count']),
                'intent_accuracy': float(intent_row['intent_accuracy']),
                'intent_macro_precision': float(intent_row['macro_precision']),
                'intent_macro_recall': float(intent_row['macro_recall']),
                'intent_macro_f1': float(intent_row['macro_f1']),
                'intent_micro_f1': float(intent_row['micro_f1']),
                'slot_macro_precision_by_intent': float(slot_row['precision']),
                'slot_macro_recall_by_intent': float(slot_row['recall']),
                'slot_macro_f1_by_intent': float(slot_row['f1']),
                'slot_micro_f1': float(slot_row['micro_f1']),
            }
        )

        intent_round = intent_by_label.copy()
        intent_round.insert(0, 'round', round_number)
        intent_by_label_frames.append(intent_round)

        slot_round = slot_by_intent.copy()
        slot_round.insert(0, 'round', round_number)
        slot_by_intent_frames.append(slot_round)

    rounds = pd.DataFrame(round_rows)
    intent_by_label_rounds = pd.concat(intent_by_label_frames, ignore_index=True)
    slot_by_intent_rounds = pd.concat(slot_by_intent_frames, ignore_index=True)

    metric_columns = [
        'intent_accuracy',
        'intent_macro_precision',
        'intent_macro_recall',
        'intent_macro_f1',
        'intent_micro_f1',
        'slot_macro_precision_by_intent',
        'slot_macro_recall_by_intent',
        'slot_macro_f1_by_intent',
        'slot_micro_f1',
    ]
    summary_rows = []
    for metric in metric_columns:
        avg, ci95_low, ci95_high = _atis_strat_mean_ci95(rounds[metric].tolist())
        summary_rows.append(
            {
                'metric': metric,
                'mean': avg,
                'ci95_low': ci95_low,
                'ci95_high': ci95_high,
                'N': n,
                'K': k,
                'seed': seed,
            }
        )
    summary = pd.DataFrame(summary_rows)

    written = {
        'summary': output_dir / 'summary.csv',
        'rounds': output_dir / 'rounds.csv',
        'coverage': output_dir / 'coverage.csv',
        'intent_by_label_rounds': output_dir / 'intent_by_label_rounds.csv',
        'slot_by_intent_rounds': output_dir / 'slot_by_intent_rounds.csv',
    }
    summary.to_csv(written['summary'], index=False, encoding='utf-8-sig')
    rounds.to_csv(written['rounds'], index=False, encoding='utf-8-sig')
    coverage.to_csv(written['coverage'], index=False, encoding='utf-8-sig')
    intent_by_label_rounds.to_csv(written['intent_by_label_rounds'], index=False, encoding='utf-8-sig')
    slot_by_intent_rounds.to_csv(written['slot_by_intent_rounds'], index=False, encoding='utf-8-sig')

    print('=' * 80)
    print(f'ATIS Stratified Macro@N: N={n}, K={k}, seed={seed}')
    print(f'Output dir: {output_dir}')
    print('Coverage by intent')
    ipy_display(coverage)
    print('Stratified summary')
    ipy_display(summary)
    print('Rounds preview')
    ipy_display(rounds.head(PREVIEW_ROWS))
    print('Arquivos exportados:')
    for name, path in written.items():
        print(f'- {name}: {path}')

    return {
        'coverage': coverage,
        'summary': summary,
        'rounds': rounds,
        'intent_by_label_rounds': intent_by_label_rounds,
        'slot_by_intent_rounds': slot_by_intent_rounds,
        'written': written,
        'output_dir': output_dir,
    }


atis_stratified_metrics = run_atis_stratified_macro_analysis()


## Field Operations

Proposito: calcular metricas apenas do Field Operations.

Pre-condicao: `evaluation/field_operations/results_csv/field_operations_predictions.csv` deve existir e corresponder a rodada que voce quer avaliar.


In [ ]:
# Celula Field Operations metrics - rode somente depois de gerar field_operations_predictions.csv.
field_operations_metrics = run_metrics_for_dataset('field_operations')
